# Can we predict failure within 90 days of an inspection?

Target: `failed_90d`. Plan: profile the data, check for leakage, evaluate with a **time-ordered split** (we would be predicting future inspections), compare a baseline against a few models, and report honestly.

In [ ]:
import numpy as np, pandas as pd, matplotlib.pyplot as plt
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.pipeline import make_pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.metrics import roc_auc_score, average_precision_score, roc_curve, precision_recall_curve
from sklearn.inspection import permutation_importance
pd.set_option('display.width', 150)
df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, df.isna().sum().sum(), 'missing;', df.asset_tag.nunique(), 'unique assets;', df.duplicated().sum(), 'duplicate rows')
print('Failure rate:', df.failed_90d.mean().round(4), '| date range', df.inspected_at.min(), '->', df.inspected_at.max())
df.describe().T

## Leakage check: `parts_replaced`
The dictionary says this is the number of parts replaced on the work order *that closed the inspection record*, i.e. recorded after the inspection and plausibly as a result of a failure/repair.

In [ ]:
df.groupby('parts_replaced').failed_90d.agg(['mean', 'size'])

Every record with 2+ parts replaced failed (100%) and none with 0 did. That is near-deterministic, so the column almost certainly reflects the outcome rather than predicting it, and it would not be available at inspection time. **It is excluded from all models.** (Worth confirming with the owner of the work-order system when this field is filled in.)

Each asset appears only once, so there is no asset-level leakage between train and test.

## Exploration

In [ ]:
m = df.groupby(df.inspected_at.dt.to_period('M')).failed_90d.agg(['mean', 'size'])
fig, ax = plt.subplots(1, 2, figsize=(12, 3.5))
m['mean'].plot(ax=ax[0], marker='o'); ax[0].set_title('Failure rate by inspection month'); ax[0].set_ylabel('rate')
df.groupby('site').failed_90d.mean().plot.bar(ax=ax[1]); ax[1].set_title('Failure rate by site')
plt.tight_layout(); plt.show()
num = ['vibration_rms', 'bearing_temp_c', 'ambient_temp_c', 'meter_hours', 'load_pct']
fig, axs = plt.subplots(1, 5, figsize=(17, 3))
for a, c in zip(axs, num):
    for y in (0, 1): a.hist(df.loc[df.failed_90d == y, c], bins=30, alpha=.5, density=True, label=str(y))
    a.set_title(c)
axs[0].legend(title='failed'); plt.tight_layout(); plt.show()

Observations: the base rate swings strongly over time (about 1-5% in some months, 30-40% in May-Aug 2024) while site makes no difference. Vibration, bearing temperature and load are higher for failures; meter hours is lower. The time swing is a caution: a model trained on one period may be miscalibrated in another, so we evaluate on the most recent period.

## Split: train on the earliest 70%, test on the latest 30%

In [ ]:
cut = int(len(df) * .7)
train, test = df.iloc[:cut], df.iloc[cut:]
print('train', train.inspected_at.min().date(), '->', train.inspected_at.max().date(), 'rate', train.failed_90d.mean().round(3))
print('test ', test.inspected_at.min().date(), '->', test.inspected_at.max().date(), 'rate', test.failed_90d.mean().round(3))
feats = num + ['site']          # parts_replaced excluded (leakage); asset_tag / time not used as features
Xtr, ytr, Xte, yte = train[feats], train.failed_90d, test[feats], test.failed_90d
pre = ColumnTransformer([('n', StandardScaler(), num), ('c', OneHotEncoder(), ['site'])])
models = {
    'logistic': make_pipeline(pre, LogisticRegression(max_iter=1000)),
    'hist_gb': make_pipeline(pre, HistGradientBoostingClassifier(max_depth=3, learning_rate=.05, max_iter=200, random_state=0)),
}
res = {}; prob = {}
for n, m in models.items():
    m.fit(Xtr, ytr); prob[n] = m.predict_proba(Xte)[:, 1]
    res[n] = dict(ROC_AUC=roc_auc_score(yte, prob[n]), PR_AUC=average_precision_score(yte, prob[n]))
res['no-skill baseline'] = dict(ROC_AUC=.5, PR_AUC=yte.mean())
pd.DataFrame(res).T.round(3)

Sanity check with the leaky column, to show how much it would inflate results (do not use it):

In [ ]:
leaky = make_pipeline(ColumnTransformer([('n', StandardScaler(), num + ['parts_replaced']), ('c', OneHotEncoder(), ['site'])]), LogisticRegression(max_iter=1000))
leaky.fit(train[feats + ['parts_replaced']], ytr)
print('ROC AUC with parts_replaced:', round(roc_auc_score(yte, leaky.predict_proba(test[feats + ['parts_replaced']])[:, 1]), 3))

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
for n, p in prob.items():
    f, t, _ = roc_curve(yte, p); ax[0].plot(f, t, label=n)
    pr, rc, _ = precision_recall_curve(yte, p); ax[1].plot(rc, pr, label=n)
ax[0].plot([0, 1], [0, 1], 'k:'); ax[0].set(title='ROC', xlabel='FPR', ylabel='TPR')
ax[1].axhline(yte.mean(), color='k', ls=':'); ax[1].set(title='Precision-recall', xlabel='recall', ylabel='precision'); ax[0].legend(); plt.tight_layout(); plt.show()

## Robustness: rolling-origin evaluation and bootstrap interval

In [ ]:
# Expanding window: train on everything before each quarter, test on that quarter
rows = []
for q in pd.period_range('2025-01', '2025-12', freq='Q'):
    te = df[df.inspected_at.dt.to_period('Q') == q]; tr = df[df.inspected_at < q.start_time]
    m = models['hist_gb'].fit(tr[feats], tr.failed_90d)
    rows.append(dict(quarter=str(q), n=len(te), rate=te.failed_90d.mean(), ROC_AUC=roc_auc_score(te.failed_90d, m.predict_proba(te[feats])[:, 1])))
display(pd.DataFrame(rows).round(3))
models['hist_gb'].fit(Xtr, ytr)   # restore the train-split fit
rng = np.random.default_rng(0); b = []
yv, pv = yte.values, prob['hist_gb']
for _ in range(1000):
    i = rng.integers(0, len(yv), len(yv)); b.append(roc_auc_score(yv[i], pv[i]))
print('Gradient-boosting test ROC AUC 95%% bootstrap CI: %.3f-%.3f' % tuple(np.percentile(b, [2.5, 97.5])))

## What drives predictions?

In [ ]:
m = models['hist_gb']
pi = permutation_importance(m, Xte, yte, scoring='roc_auc', n_repeats=20, random_state=0)
pd.Series(pi.importances_mean, index=feats).sort_values().plot.barh(title='Permutation importance (test ROC AUC drop)'); plt.show()
lr = models['logistic']; print(pd.Series(lr[-1].coef_[0], index=lr[0].get_feature_names_out()).round(2).to_string())

## Operating point: flagging the top 10% riskiest inspections

In [ ]:
p = prob['hist_gb']; k = int(.1 * len(p)); top = np.argsort(-p)[:k]
print(f'Flag top 10%: precision {yte.values[top].mean():.2f} (base rate {yte.mean():.2f}), recall {yte.values[top].sum()/yte.sum():.2f}, lift {yte.values[top].mean()/yte.mean():.1f}x')

## Conclusion
- **Yes, but only modestly.** On the most recent period (Jun-Dec 2025, 5.6% failure rate) gradient boosting reaches ROC AUC about 0.76 (bootstrap 95% CI about 0.71-0.80) and PR AUC about 0.17 versus 0.056 for no skill; logistic regression is weaker (about 0.68), so some non-linearity matters. Quarterly rolling-origin AUCs are 0.73-0.84.
- Flagging the top 10% of inspections catches roughly 29% of failures at about 16% precision (about 2.9x lift). Useful for prioritising inspections, not for confident individual calls.
- Signal comes from vibration, bearing temperature and load (higher means riskier) and meter hours / ambient temperature; site is uninformative.
- **`parts_replaced` is excluded as leakage.** Including it gives AUC about 0.997, which would be misleading.
- **Caveats:** the base rate drifts a lot over time (about 1% to 39% by month), so probabilities need recalibration before use as absolute risks. Low rates at the end of 2025 might also reflect unobserved failures if the data was cut before the 90-day windows closed. Please confirm how `failed_90d` was labelled for late-2025 inspections. No hyperparameter tuning was done.
- The saved notebook has no stored outputs (the Jupyter kernel could not start in my sandbox); run all cells to populate them.